# Prophet: Generate Predictions for All Models

This notebook loads outcome prediction data and generates predictions using all available disease models in Prophet.

In [ ]:
import sys
import os
from pathlib import Path
import logging
from datetime import datetime

import polars as pl

# Add prophet to path
prophet_root = Path('/home/niels/Desktop/prophet')
sys.path.insert(0, str(prophet_root / 'src'))

from prophet import Prophet

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration

In [ ]:
# Data configuration
DATA_DIR = '/home/niels/Desktop/outcome_nlp'
OUTPUT_DIR = '/home/niels/Desktop/prophet/results/predictions'

# Prediction settings
SHOW_PROGRESS = True
RETURN_FEATURES = False  # Set to True to also save feature matrices

# Optional: specify which models to run (leave as None to run all)
MODELS_TO_RUN = None  # or e.g., ['stroke', 'dementia', 'cardiac'] to run specific models

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Show progress: {SHOW_PROGRESS}")
print(f"Return features: {RETURN_FEATURES}")

## Load Data

In [ ]:
# Map outcome file suffixes to internal names
DATA_FILE_MAPPING = {
    'notes': 'notes',
    'icd': 'icd',
    'med': 'med',
    'cpt': 'cpt',
    'demo': 'demo'
}

data = {}
for suffix, name in DATA_FILE_MAPPING.items():
    file_path = os.path.join(DATA_DIR, f'outcome_prediction_{suffix}.parquet')
    
    if not os.path.exists(file_path):
        print(f"⚠ File not found: {file_path}")
        continue
    
    df = pl.read_parquet(file_path)
    data[name] = df
    print(f"✓ Loaded {name:8s}: {len(df):>8,} rows, {len(df.columns):>3} columns")

print(f"\nTotal data sources loaded: {len(data)}")
print(f"Data keys: {list(data.keys())}")

## Initialize Prophet and Check Available Models

In [ ]:
# Initialize Prophet
prophet = Prophet()

# Get all available models
available_models = prophet.get_available_models()
print(f"Total available models: {len(available_models)}")
print(f"\nAll models:")
for i, model in enumerate(sorted(available_models), 1):
    print(f"  {i:2d}. {model}")

## Find Compatible Models

In [ ]:
# Find compatible models
compatible_models = prophet.find_compatible_models(data)
print(f"Compatible models with provided data: {len(compatible_models)}")
print(f"\nCompatible models:")
for i, model in enumerate(sorted(compatible_models), 1):
    print(f"  {i:2d}. {model}")

incompatible = set(available_models) - set(compatible_models)
if incompatible:
    print(f"\nIncompatible models ({len(incompatible)}):")
    for i, model in enumerate(sorted(incompatible), 1):
        print(f"  {i:2d}. {model}")

## Run Predictions

In [ ]:
# Determine which models to run
if MODELS_TO_RUN is None:
    models_to_run = compatible_models
else:
    # Filter to only models that are both requested and compatible
    models_to_run = [m for m in MODELS_TO_RUN if m in compatible_models]
    if not models_to_run:
        print(f"Error: No requested models are compatible")
        print(f"Requested: {MODELS_TO_RUN}")
        print(f"Compatible: {compatible_models}")
    else:
        skipped = [m for m in MODELS_TO_RUN if m not in compatible_models]
        if skipped:
            print(f"Skipping incompatible models: {skipped}")

print(f"Running predictions for {len(models_to_run)} models")
print(f"Starting at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

In [ ]:
# Run predictions for all compatible models
results = {}

for i, model_name in enumerate(sorted(models_to_run), 1):
    try:
        print(f"[{i:2d}/{len(models_to_run)}] Running '{model_name}'...", end=' ', flush=True)
        result = prophet.predict(
            model_name,
            data=data,
            show_progress_bar=SHOW_PROGRESS,
            return_features=RETURN_FEATURES
        )
        results[model_name] = result
        print(f"✓")
    except Exception as e:
        print(f"✗ Error: {e}")
        continue

print(f"\nSuccessfully generated predictions for {len(results)} models")
print(f"Failed: {len(models_to_run) - len(results)} models")

## Save Results

In [ ]:
# Save predictions
print(f"Saving predictions to {OUTPUT_DIR}")
prophet.save_predictions(results, OUTPUT_DIR)

# List saved files
saved_files = []
for model_name in results.keys():
    model_dir = os.path.join(OUTPUT_DIR, model_name)
    if os.path.exists(model_dir):
        files = os.listdir(model_dir)
        saved_files.append((model_name, files))

print(f"\nSaved results for {len(saved_files)} models:")
for model_name, files in sorted(saved_files):
    print(f"\n  {model_name}/")
    for f in sorted(files):
        path = os.path.join(OUTPUT_DIR, model_name, f)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"    - {f} ({size_mb:.2f} MB)")

## Summary

In [ ]:
print(f"\nPipeline completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nResults summary:")
print(f"  Total models available:    {len(available_models)}")
print(f"  Compatible models:          {len(compatible_models)}")
print(f"  Models run successfully:    {len(results)}")
print(f"  Output directory:           {OUTPUT_DIR}")